# Deep learning: frozen embeddings and fine-tuned DistilBERT (notebook 03)

Session 7 order: first freeze a pretrained encoder and train only a small head on its
embeddings (cheap, cached), then fine-tune the whole encoder (expensive). Each model is scored
once on test per (model, seed) on the same five splits as notebook 02; comparing and plotting
the saved results is notebook 04. Heavy cells need a GPU and the `torch`, `transformers` and
`sentence-transformers` packages; the run cells are skipped unless `ON_GPU_SERVER = True`
(set it on the server only).

Run time: one DistilBERT run took about 7 minutes on one GPU at batch size 128 (measured, seed 0,
5 epochs); the embedding heads take seconds once the embeddings are cached.

In [ ]:
import sys, os; os.chdir(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else None; sys.path.insert(0, "."); import warnings
import importlib.util

from modules import data_helper as dh
from modules import features as ft
from modules import deep_runs as dr

DATA = "../data/fake_job_postings.csv"

In [2]:
SEED = 0
SEEDS = [0, 1, 2, 3, 4]
BERT_SEEDS = SEEDS  # seeds with existing result files are skipped
OUT_DEEP = "outputs/deep"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BERT_MODEL = "distilbert-base-uncased"
EPOCHS, BATCH_SIZE, MAX_LEN, LR = 5, 128, 256, 2e-5
LOG_EVERY = 10  # print the loss every this many steps
DEVICE = "cuda:1"  # which GPU to use; the server is shared, so pick an emptier one (nvidia-smi), e.g. "cuda:1"

# SAFETY: the embedding and fine-tuning cells are heavy GPU work. They run only when this flag is
# True, and it must stay False on the laptop. Set it to True on the GPU server, nowhere else.
ON_GPU_SERVER = True

# Libraries present? (a missing one would make the heavy cells fail, not skip)
HAVE_TORCH = all(importlib.util.find_spec(m) for m in ("torch", "transformers", "sentence_transformers"))
HAVE_EMBEDDINGS = os.path.exists("features/" + EMBED_MODEL.replace("/", "_") + "_emb.npy")

# Heavy cells need the explicit opt-in AND the libraries; the sklearn heads are light (CPU) and
# only need the cached embeddings.
RUN_EMBED = RUN_BERT = ON_GPU_SERVER and HAVE_TORCH
RUN_HEADS = RUN_EMBED or HAVE_EMBEDDINGS
print({"ON_GPU_SERVER": ON_GPU_SERVER, "torch stack": HAVE_TORCH, "cached embeddings": HAVE_EMBEDDINGS})

dh.seed_everything(SEED)

{'ON_GPU_SERVER': True, 'torch stack': True, 'cached embeddings': True}


## Data and splits

The same cleaned frame and the same saved splits as the classical notebook, so every model
is scored on identical rows.

In [3]:
df = ft.add_engineered_features(dh.clean(dh.load_raw(DATA)))
print("rows:", len(df), "| fraud:", int(df["fraudulent"].sum()))

rows: 17599 | fraud: 856


## Step 1: frozen sentence embeddings

`all-MiniLM-L6-v2` turns each ad (the joined `text` column) into one 384-dimensional vector.
Nothing is trained here. The vectors are cached in `features/` keyed by `job_id`, so this
cell is a no-op the second time.

In [ ]:
if RUN_EMBED:
    # Encode every ad once; the result is cached in features/.
    dr.run_embed(df, DEVICE, EMBED_MODEL)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + sentence-transformers")

## Step 2: sklearn heads on the frozen embeddings

A logistic regression (C picked on validation PR-AUC) and a small MLP, trained on the cached
vectors with CPU only. Each is run for all five seeds; the threshold is picked on validation and
the test rows are scored once.

In [5]:
if RUN_HEADS:
    dr.run_heads(df, SEEDS, OUT_DEEP, EMBED_MODEL)
else:
    print("skipped: no cached embeddings here")

embeddings: (17599, 384)
seed 0:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
emb_logreg seed 0: val PR-AUC 0.576, test PR-AUC 0.506
emb_mlp seed 0: val PR-AUC 0.702, test PR-AUC 0.636
seed 1:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
emb_logreg seed 1: val PR-AUC 0.624, test PR-AUC 0.522
emb_mlp seed 1: val PR-AUC 0.698, test PR-AUC 0.527
seed 2:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
emb_logreg seed 2: val PR-AUC 0.604, test PR-AUC 0.436
emb_mlp seed 2: val PR-AUC 0.728, test PR-AUC 0.639
seed 3:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
emb_logreg seed 3: val PR-AUC 0.614, test PR-AUC 0.592
emb_mlp seed 3: val PR-AUC 0.738, test PR-AUC 0.776

## Step 3: fine-tune DistilBERT

The whole encoder is trained with a linear classification layer on top: AdamW, learning rate
2e-5 with warm-up and linear decay, square-root class weights in the loss, bf16 autocast on
CUDA, and early stopping on validation PR-AUC (the best epoch's weights are restored). Texts
are truncated to 256 tokens. `BERT_SEEDS` lists the seeds to fine-tune; finished ones are skipped.

In [6]:
if RUN_BERT:
    dr.run_bert(df, BERT_SEEDS, OUT_DEEP, DEVICE, BERT_MODEL, epochs=EPOCHS,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, lr=LR, log_every=LOG_EVERY)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers")

seed 0:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
loading distilbert-base-uncased (downloads on the first run)...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.6763
  step 10/83  loss 0.6271
  step 20/83  loss 0.5193
  step 30/83  loss 0.4270
  step 40/83  loss 0.4045
  step 50/83  loss 0.4676
  step 60/83  loss 0.4090
  step 70/83  loss 0.4044
  step 80/83  loss 0.1150
epoch 1/5: train loss 0.4175, val PR-AUC 0.606, 73s
  step 0/83  loss 0.2961
  step 10/83  loss 0.1336
  step 20/83  loss 0.1019
  step 30/83  loss 0.1110
  step 40/83  loss 0.3481
  step 50/83  loss 0.0987
  step 60/83  loss 0.1842
  step 70/83  loss 0.0625
  step 80/83  loss 0.1775
epoch 2/5: train loss 0.1657, val PR-AUC 0.808, 72s
  step 0/83  loss 0.0740
  step 10/83  loss 0.0531
  step 20/83  loss 0.3184
  step 30/83  loss 0.0560
  step 40/83  loss 0.0834
  step 50/83  loss 0.0416
  step 60/83  loss 0.0285
  step 70/83  loss 0.1777
  step 80/83  loss 0.1137
epoch 3/5: train loss 0.0907, val PR-AUC 0.845, 72s
  step 0/83  loss 0.0232
  step 10/83  loss 0.0130
  step 20/83  loss 0.0366
  step 30/83  loss 0.3432
  step 40/83 